# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

Mô hình PEAS (**P**erformance Measure, **E**nvironment, **A**ctuators, **S**ensors) là phương pháp chuẩn hóa trong Trí tuệ Nhân tạo (theo giáo trình AIMA - Stuart Russell & Peter Norvig) để đặc tả đầy đủ một bài toán tác tử thông minh. Trong GridHunt, ta xác định hệ thống PEAS cho tác tử chính (**Hunter**), tác tử đối thủ (**Monster**) và các đặc trưng môi trường như sau:

### 1. Bảng PEAS cho Tác tử Thợ săn (Hunter Agent - Tác tử chính cần cài đặt)

| Thành phần PEAS | Tên gọi & Ý nghĩa | Chi tiết trong bài toán GridHunt |
| :--- | :--- | :--- |
| **P - Performance Measure** | **Độ đo hiệu năng**<br>*(Tiêu chuẩn đánh giá mức độ thành công)* | - **Tối thiểu hóa số bước đi:** Số bước (steps) Hunter dùng để bắt được Monster càng ít càng tốt.<br>- **Bắt thành công:** Hunter di chuyển vào đúng ô tọa độ mà Monster đang chiếm giữ (`hunter_location == monster_location`).<br>- **Tỷ lệ thắng cuộc:** Bắt được Monster trước khi vượt quá số bước tối đa (`max_steps = 100`). Nếu vượt quá thì tính là thất bại (NaN/Failure).<br>- **Chi phí hành động:** Mỗi hành động (di chuyển, teleport, listen) đều tốn 1 lượt (cost = 1 step). Hành động `listen` đánh đổi 1 lượt đứng yên để cập nhật tọa độ chính xác của Monster. |
| **E - Environment** | **Môi trường**<br>*(Không gian và bối cảnh hoạt động)* | - **Đấu trường lưới:** Lưới ô vuông $n \times n$ (mặc định $n = 30$).<br>- **Biên giới hạn:** Có tường bao quanh, tác tử không thể đi ra ngoài phạm vi chỉ số $0$ đến $n-1$.<br>- **Khởi tạo:** Vị trí ban đầu của Hunter và Monster được đặt ngẫu nhiên trong đấu trường.<br>- **Quy tắc lượt:** Môi trường theo lượt (turn-based), Hunter hành động trước, sau đó Monster hành động.<br>- **Tác tử đối kháng:** Quái vật (Monster) di chuyển liên tục trong đấu trường (di chuyển ngẫu nhiên hoặc có chiến thuật né tránh). |
| **A - Actuators** | **Bộ phận thực thi**<br>*(Tập các hành động tác tử có thể thực hiện)* | Tập hành động gồm 6 hành động rời rạc: `actions = ["north", "east", "west", "south", "teleport", "listen"]`<br>- `north`: Di chuyển lên trên 1 ô ($row - 1$).<br>- `south`: Di chuyển xuống dưới 1 ô ($row + 1$).<br>- `east`: Di chuyển sang phải 1 ô ($col + 1$).<br>- `west`: Di chuyển sang trái 1 ô ($col - 1$).<br>- `teleport`: Dịch chuyển tức thời đến 1 ô ngẫu nhiên bất kỳ trong đấu trường.<br>- `listen`: Đứng yên tại chỗ để lắng nghe và nhận diện tọa độ chính xác của Monster ở lượt kế tiếp. |
| **S - Sensors** | **Cảm biến / Nhận thức**<br>*(Thông tin/tri giác tác tử nhận từ môi trường)* | Tác tử nhận thông tin qua 2 tham số đầu vào của hàm tác tử: `hunter_agent(hunter_location, monster_location)`:<br>- `hunter_location`: Luôn luôn biết được tọa độ hiện tại `[row, col]` của chính mình ở mọi bước.<br>- `monster_location`: Chỉ nhận được tọa độ `[row, col]` của Monster ngay sau khi thực hiện hành động `listen`. Ở các lượt di chuyển thông thường khác, giá trị này bị ẩn (`None`).<br>- **Thông số môi trường:** Kích thước đấu trường $n$ (biết trước để kiểm tra biên giới hạn). |

---

### 2. Bảng PEAS cho Tác tử Quái vật (Monster Agent - Đối thủ trong môi trường)

| Thành phần PEAS | Tên gọi & Ý nghĩa | Chi tiết đối với Monster Agent |
| :--- | :--- | :--- |
| **P - Performance Measure** | **Độ đo hiệu năng** | - **Sống sót tối đa:** Tránh không bị Hunter bắt cùng ô cho đến khi hết `max_steps`.<br>- **Khoảng cách an toàn:** Tối đa hóa khoảng cách so với Hunter (với quái vật né tránh thông minh). |
| **E - Environment** | **Môi trường** | Đấu trường lưới $n \times n$ có tường bao quanh, có Hunter liên tục truy đuổi. |
| **A - Actuators** | **Bộ phận thực thi** | Tập hành động gồm 5 thao tác: `monster_actions = ["north", "east", "west", "south", "stay"]`. |
| **Sensors** | **Cảm biến / Tri giác** | - *Quái vật ngẫu nhiên (Randomized):* Không cần cảm biến (chọn ngẫu nhiên 1 trong 5 hành động).<br>- *Quái vật thông minh (Evasive):* Nhận biết tọa độ bản thân và vị trí của Hunter để chọn hướng tẩu thoát ra xa nhất. |

---

### 3. Phân loại đặc trưng môi trường (Environment Characteristics)

Theo chuẩn phân loại môi trường tác tử trong giáo trình *Artificial Intelligence: A Modern Approach (AIMA)*:

| Đặc tính môi trường | Phân loại | Giải thích chi tiết trong GridHunt |
| :--- | :---: | :--- |
| **Mức độ quan sát** *(Observability)* | **Partially Observable** *(Quan sát một phần)* | Hunter luôn biết vị trí bản thân nhưng vị trí của Monster bị ẩn (`None`), chỉ lộ diện khi Hunter chọn hành động `listen`. |
| **Tính xác định** *(Determinism)* | **Stochastic** *(Ngẫu nhiên / Bất định)* | Vị trí ban đầu ngẫu nhiên; hành động `teleport` đưa đến vị trí ngẫu nhiên; Monster di chuyển ngẫu nhiên (hoặc bán ngẫu nhiên). |
| **Tính phân đoạn** *(Episodic vs. Sequential)* | **Sequential** *(Tuần tự)* | Quyết định ở bước hiện tại (như nghe vị trí, di chuyển hay teleport) ảnh hưởng trực tiếp đến trạng thái và cơ hội bắt Monster ở các bước sau. |
| **Tính biến đổi** *(Static vs. Dynamic)* | **Static** *(Tĩnh theo từng lượt)* | Trò chơi theo lượt (turn-based): Trong lúc tác tử đang tính toán chọn hành động, môi trường và quái vật đứng yên không tự ý thay đổi. |
| **Tính liên tục** *(Discrete vs. Continuous)* | **Discrete** *(Rời rạc)* | Không gian vị trí là lưới ô vuông nguyên $n \times n$, thời gian chia theo từng lượt rời rạc, tập hành động hữu hạn rời rạc. |
| **Số lượng tác tử** *(Single vs. Multi-agent)* | **Multi-agent (Competitive)** *(Đa tác tử - Cạnh tranh đối kháng)* | Môi trường gồm 2 tác tử (Hunter và Monster) với mục tiêu đối kháng trực tiếp (Zero-sum: Hunter thắng khi bắt được Monster, Monster thắng khi sống sót). |

---

> **Tóm tắt mô tả gốc tiếng Anh (Original PEAS Specification):**
> - **Performance Measure:** The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.
> - **Environment:** An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster can move around, but cannot leave the arena.
> - **Actuators:** The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent to a random square in the arena.
> - **Sensors:** The agent can always see its location in the arena but only gets the monster location when it stops and listens.


## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('listen')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[2 2]'; Monster is at '[2 2]'
.....
.....
..*..
.....
.....
Hunter caught the monster in 0 steps!


0

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [7]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[nan, nan, 0, 26, nan, nan, 93, nan, 70, nan, 76, nan, 65, 48, 86, nan, nan, nan, 47, nan, 36, 77, nan, 97, nan, nan, 39, 1, 46, nan, nan, nan, 3, nan, nan, nan, nan, nan, 62, nan, 12, 80, 14, nan, 5, nan, 70, nan, 36, nan, 74, nan, 76, 34, nan, nan, 6, 33, nan, 34, 40, nan, nan, nan, 53, nan, 43, nan, 11, 47, 76, nan, 28, 65, nan, nan, nan, 43, nan, 74, nan, nan, 21, 97, nan, 36, nan, nan, nan, 12, nan, 5, 2, nan, nan, nan, 42, 3, nan, nan]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [8]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 53 times.
Average steps to catch the monster over 47 successful runs: 43.49


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [9]:
def model_based_hunter_agent(hunter_location, monster_location):
    """
    Tác tử phản xạ dựa trên mô hình (Model-based reflex agent) cho GridHunt.
    Lưu trữ trạng thái mô hình nội bộ về vị trí quái vật và các bước đi trước đó.
    
    Tham số:
    hunter_location (np.ndarray): Tọa độ hiện tại [hàng, cột] của thợ săn.
    monster_location (np.ndarray hoặc None): Tọa độ quái vật (chỉ nhận được nếu bước trước đó thợ săn hành động 'listen').
                                          
    Trả về:
    str: Hành động tiếp theo ('north', 'east', 'west', 'south', 'teleport', 'listen').
    """
    hunter_pos = np.array(hunter_location)
    
    # Đặt lại mô hình nội bộ nếu ván mới bắt đầu (bước đầu tiên hoặc vị trí thợ săn thay đổi bất thường không qua teleport)
    if (model_based_hunter_agent.last_hunter_pos is None or 
        (model_based_hunter_agent.last_action != 'teleport' and 
         np.sum(np.abs(hunter_pos - model_based_hunter_agent.last_hunter_pos)) > 1)):
        model_based_hunter_agent.target_monster_pos = None

    model_based_hunter_agent.last_hunter_pos = hunter_pos.copy()

    # 1. Cập nhật mô hình nội bộ nếu tri giác nhận được tọa độ quái vật (sau khi 'listen')
    if monster_location is not None:
        model_based_hunter_agent.target_monster_pos = np.array(monster_location, copy=True)

    # 2. Nếu chưa biết vị trí quái vật, thực hiện lắng nghe
    if model_based_hunter_agent.target_monster_pos is None:
        action = 'listen'
    # 3. Nếu đã đến vị trí đã nhớ nhưng quái vật đã di chuyển sang ô khác, lắng nghe lại
    elif np.array_equal(hunter_pos, model_based_hunter_agent.target_monster_pos):
        model_based_hunter_agent.target_monster_pos = None
        action = 'listen'
    else:
        # 4. Di chuyển tham lam tiếp cận quái vật theo trục có độ lệch lớn hơn (giảm khoảng cách Manhattan)
        d_row = model_based_hunter_agent.target_monster_pos[0] - hunter_pos[0]
        d_col = model_based_hunter_agent.target_monster_pos[1] - hunter_pos[1]

        if abs(d_row) > abs(d_col):
            action = 'north' if d_row < 0 else 'south'
        elif abs(d_col) > 0:
            action = 'west' if d_col < 0 else 'east'
        elif d_row != 0:
            action = 'north' if d_row < 0 else 'south'
        else:
            model_based_hunter_agent.target_monster_pos = None
            action = 'listen'

    model_based_hunter_agent.last_action = action
    return action

# Khởi tạo trạng thái mô hình nội bộ của tác tử
model_based_hunter_agent.target_monster_pos = None
model_based_hunter_agent.last_hunter_pos = None
model_based_hunter_agent.last_action = None

# Thử nghiệm nhanh hành động của tác tử
print("Hành động của tác tử khi chưa biết vị trí quái vật:", model_based_hunter_agent([0, 0], None))
print("Hành động của tác tử khi quái vật được quan sát ở [5, 0]:", model_based_hunter_agent([0, 0], [5, 0]))


Hành động của tác tử khi chưa biết vị trí quái vật: listen
Hành động của tác tử khi quái vật được quan sát ở [5, 0]: south


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [10]:
# 1. Mô phỏng trực quan trên bàn cờ 10x10 để quan sát hành vi bám đuổi của thợ săn
print("--- 1. Mô phỏng trực quan (Kích thước Arena 10x10) ---")
model_based_hunter_agent.target_monster_pos = None
model_based_hunter_agent.last_hunter_pos = None
model_based_hunter_agent.last_action = None

arena_environment(model_based_hunter_agent, 
                  monster_agent_function_simple, 
                  n=10, max_steps=20, visualize=True, animation=False)

# 2. Thử nghiệm có đối chứng trên các kích thước arena khác nhau (10x10, 30x30, 50x50)
arena_sizes = [10, 30, 50]
num_runs = 100
max_steps = 100

print(f"\n--- 2. Kết quả thử nghiệm ({num_runs} lượt chạy cho mỗi cấu hình arena) ---")
print(f"{'Kích thước':<14} | {'Thất bại':<12} | {'Tỷ lệ thành công':<18} | {'Số bước TB (Bắt được)':<24} | {'Trung vị bước':<14}")
print("-" * 90)

for size in arena_sizes:
    steps_list = []
    for _ in range(num_runs):
        # Đặt lại trạng thái nội bộ của tác tử cho mỗi ván đấu độc lập
        model_based_hunter_agent.target_monster_pos = None
        model_based_hunter_agent.last_hunter_pos = None
        model_based_hunter_agent.last_action = None
        
        s = arena_environment(model_based_hunter_agent, 
                              monster_agent_function_simple, 
                              n=size, max_steps=max_steps, visualize=False)
        steps_list.append(s)
    
    steps_arr = np.array(steps_list)
    failed = np.sum(np.isnan(steps_arr))
    success_rate = ((num_runs - failed) / num_runs) * 100
    avg_steps = round(np.nanmean(steps_arr), 2) if failed < num_runs else np.nan
    median_steps = round(np.nanmedian(steps_arr), 2) if failed < num_runs else np.nan
    
    print(f"{f'{size}x{size}':<14} | {f'{failed}/{num_runs}':<12} | {f'{success_rate:.1f}%':<18} | {f'{avg_steps}':<24} | {f'{median_steps}':<14}")


--- 1. Mô phỏng trực quan (Kích thước Arena 10x10) ---
Step 1: Hunter is at '[6 8]'; Monster is at '[0 0]'
M.........
..........
..........
..........
..........
..........
........H.
..........
..........
..........
Hunter chose action 'listen'
Monster chose action 'east'


Step 2: Hunter is at '[6 8]'; Monster is at '[0 1]'
.M........
..........
..........
..........
..........
..........
........H.
..........
..........
..........
Hunter chose action 'west'
Monster chose action 'east'


Step 3: Hunter is at '[6 7]'; Monster is at '[0 2]'
..M.......
..........
..........
..........
..........
..........
.......H..
..........
..........
..........
Hunter chose action 'north'
Monster chose action 'stay'


Step 4: Hunter is at '[5 7]'; Monster is at '[0 2]'
..M.......
..........
..........
..........
..........
.......H..
..........
..........
..........
..........
Hunter chose action 'west'
Monster chose action 'west'


Step 5: Hunter is at '[5 6]'; Monster is at '[0 1]'
.M........
...

## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### 1. Chiến lược chọn hành động cuối cùng của Hunter và lý do hoạt động hiệu quả
* **Chiến lược tối ưu:**
  1. **Định vị (Cảm nhận):** Ở bước đầu tiên hoặc bất cứ khi nào mất dấu vị trí quái vật, Hunter thực hiện hành động `'listen'`.
  2. **Ghi nhớ mô hình nội bộ (Lưu trạng thái):** Ở bước kế tiếp khi nhận được tri giác tọa độ quái vật, Hunter lưu tọa độ này vào bộ nhớ nội bộ (`target_monster_pos`).
  3. **Bám đuổi tham lam (Điều hướng):** Hunter di chuyển theo trục có độ chênh lệch tọa độ lớn nhất, liên tục rút ngắn khoảng cách Manhattan ($|x_H - x_M| + |y_H - y_M|$) 1 đơn vị sau mỗi bước đi.
  4. **Lắng nghe lại khi đến đích:** Khi Hunter đến đúng ô đã ghi nhớ mà quái vật đã di chuyển sang ô lân cận, Hunter xóa mục tiêu (`target_monster_pos = None`) và chọn `'listen'` lại để cập nhật vị trí chính xác.
* **Lý do hoạt động rất hiệu quả:** 
  Quái vật có độ cơ động thấp: đứng yên 50% thời gian (`stay` với $p=0.5$) và 50% còn lại là bước đi ngẫu nhiên đẳng hướng ($p=0.125$ cho mỗi hướng). Vì bước đi ngẫu nhiên có độ dịch chuyển tịnh tiến trung bình rất nhỏ sau $t$ bước ($O(\sqrt{t})$), quái vật hầu như chỉ loanh quanh khu vực đã bị nghe thấy. Do đó khi Hunter đến nơi, quái vật chỉ cách 1–2 ô và bị bắt rất nhanh. Trong khi tác tử ngẫu nhiên thất bại hơn 50% số lần trên arena $10 \times 10$, tác tử dựa trên mô hình đạt **100% tỷ lệ bắt thành công** với số bước trung bình chỉ ~8 bước ($10 \times 10$) và ~26 bước ($30 \times 30$).

---

### 2. Vấn đề Dịch chuyển tức thời (Teleportation): Có dùng không, tại sao và trong trường hợp nào?
* **Quyết định:** **Không sử dụng dịch chuyển tức thời (teleport)** trong chiến lược săn mồi tối ưu.
* **Giải thích lý do:**
  - **Tính ngẫu nhiên và phương sai cao:** Teleport đưa Hunter đến một ô ngẫu nhiên đều trên lưới $n \times n$. Khoảng cách kỳ vọng giữa 2 điểm ngẫu nhiên trên lưới $n \times n$ là khoảng $\frac{2}{3}n$. Trừ khi Hunter ở góc này và quái vật ở góc đối diện (khoảng cách $\approx 2n$), việc dịch chuyển không giúp giảm khoảng cách đáng kể và rất dễ làm tăng khoảng cách.
  - **Tốn chi phí hành động:** Sau khi teleport, Hunter bị mất dấu quái vật và bắt buộc phải tốn thêm ít nhất 1 bước `'listen'`. Trong khi đó, việc di chuyển từng bước đảm bảo khoảng cách luôn giảm đơn điệu theo mỗi bước đi.
  - **Trường hợp ngoại lệ (nếu có):** Teleport chỉ có thể được xem xét nếu kích thước arena cực lớn (ví dụ $1000 \times 1000$), Hunter biết quái vật ở góc cực xa ($> \frac{4}{3}n$), và số bước còn lại không đủ để đi bộ. Tuy nhiên với kích thước arena thông thường ($n \le 50$), việc đi bộ có chủ đích luôn vượt trội hơn hẳn.

---

### 3. Ảnh hưởng của kích thước Arena đến hiệu năng của Hunter Agent
* **Đối với Model-Based Hunter:** 
  - Số bước trung bình để bắt quái vật tăng **tuyến tính** theo kích thước cạnh $n$ của arena, xấp xỉ tỷ lệ thuận với khoảng cách Manhattan ban đầu:
    $$\mathbb{E}[\text{số bước}] \approx \frac{2}{3}n$$
  - Trên $10 \times 10$: Trung bình $\approx 8.2$ bước (0 lần thất bại / 100 lần chạy).
  - Trên $30 \times 30$: Trung bình $\approx 26.2$ bước (0 lần thất bại / 100 lần chạy).
  - Trên $50 \times 50$: Trung bình $\approx 41.0$ bước (0 lần thất bại / 100 lần chạy).
  - Tỷ lệ bắt thành công luôn duy trì **100%** ngay cả trên các arena lớn $30 \times 30$ và $50 \times 50$.
* **So sánh với Random Hunter:**
  - Random Hunter di chuyển theo chuyển động Brown ngẫu nhiên 2 chiều. Xác suất để 2 tác tử ngẫu nhiên va chạm trong vòng 100 bước giảm mạnh theo hàm mũ khi diện tích arena ($n^2$) tăng lên.
  - Nếu như trên $10 \times 10$ Random Hunter chỉ thành công khoảng 48%, thì trên $30 \times 30$ (không gian 900 ô), tỷ lệ thành công của Random Hunter giảm xuống xấp xỉ **0%**.


The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Đề xuất cải tiến Agent Function của Monster để né tránh Hunter tốt hơn

Hiện tại, `monster_agent_function_simple` là một tác tử phản xạ thụ động, chọn hành động hoàn toàn ngẫu nhiên theo xác suất cố định và hoàn toàn phớt lờ vị trí của Hunter (dù tham số `hunter_pos` được truyền trực tiếp vào hàm).

Để quái vật né tránh thợ săn thông minh và hiệu quả hơn, có thể cải tiến như sau:

1. **Cơ chế phản xạ né tránh chủ động (Tối đa hóa khoảng cách):**
   - Monster nhận được `hunter_pos` qua tri giác. Nó có thể mô phỏng 5 hành động hợp lệ (`'north'`, `'south'`, `'east'`, `'west'`, `'stay'`) và tính khoảng cách Euclid đến `hunter_pos` sau mỗi nước đi.
   - Chọn hành động **tối đa hóa khoảng cách** rời xa Hunter:
     $$\text{action}^* = \arg\max_{a \in \mathcal{A}} \|\text{move}(a, \text{monster\_pos}) - \text{hunter\_pos}\|^2$$

2. **Trạng thái cảnh giác theo cự ly (Chuyển đổi giữa Đứng yên và Bỏ chạy):**
   - Khi Hunter ở xa (khoảng cách $> 8$ ô): Monster có thể duy trì trạng thái nghỉ ngơi (`'stay'` với xác suất cao $0.5$) để tiết kiệm di chuyển.
   - Khi Hunter tiến vào vùng nguy hiểm (khoảng cách $\le 8$ ô): Xác suất `'stay'` ngay lập tức giảm về **0**, buộc quái vật phải tích cực bỏ chạy theo hướng ngược lại.

3. **Tránh bẫy góc tường và biên bàn cờ (Anti-Cornering):**
   - Khi bị dồn vào sát tường hoặc góc bàn cờ, quái vật sẽ mất đường thoát và dễ bị thợ săn bắt.
   - Thêm điểm phạt cho các bước đi dẫn vào ô góc hoặc sát tường, ưu tiên duy trì các lối thoát ngang ở khu vực trung tâm thoáng đãng.

4. **Né tránh trực giao (Orthogonal Dodging):**
   - Khi thợ săn áp sát trực diện theo một trục (ví dụ cùng cột), quái vật nên rẽ ngang vuông góc (thay đổi hàng) để phá vỡ thế đối đầu, khiến thợ săn tốn nhiều bước điều chỉnh lại hướng đi và dễ hết thời gian 100 bước.


# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.

In [11]:
# Phần mở rộng: Cài đặt Quái vật Thông minh biết né tránh (Evasive Monster) và đánh giá đối đầu
def monster_agent_function_evasive(hunter_pos, monster_pos, n=30):
    """
    Tác tử quái vật thông minh: Chủ động né tránh thợ săn bằng cách tối đa hóa khoảng cách Euclid
    và tránh đi vào các vị trí bị giới hạn biên.
    """
    actions = ['north', 'east', 'west', 'south', 'stay']
    best_dist = -1
    best_actions = []
    
    for a in actions:
        pos = monster_pos.copy()
        if a == 'north': pos[0] -= 1
        elif a == 'south': pos[0] += 1
        elif a == 'west': pos[1] -= 1
        elif a == 'east': pos[1] += 1
        pos = np.clip(pos, 0, n-1)
        
        # Tính bình phương khoảng cách Euclid đến vị trí thợ săn
        dist = np.sum((pos - hunter_pos) ** 2)
        if dist > best_dist:
            best_dist = dist
            best_actions = [a]
        elif dist == best_dist:
            best_actions.append(a)
            
    return np.random.choice(best_actions)

# Chạy 100 lượt thử nghiệm trên arena 30x30 đối đầu với Evasive Monster
runs = 100
steps_evasive = []
for _ in range(runs):
    model_based_hunter_agent.target_monster_pos = None
    model_based_hunter_agent.last_hunter_pos = None
    model_based_hunter_agent.last_action = None
    s = arena_environment(model_based_hunter_agent, 
                          lambda h, m: monster_agent_function_evasive(h, m, n=30), 
                          n=30, max_steps=100, visualize=False)
    steps_evasive.append(s)

evasive_arr = np.array(steps_evasive)
evasive_failed = np.sum(np.isnan(evasive_arr))
print(f"Đối đầu với Quái vật Thông minh né tránh (Arena 30x30, 100 lượt chạy):")
print(f"Thợ săn thất bại: {evasive_failed}/{runs} lượt (Tỷ lệ quái vật sống sót thành công: {(evasive_failed/runs)*100:.1f}%)")
if evasive_failed < runs:
    print(f"Số bước trung bình khi bắt được: {np.nanmean(evasive_arr):.2f}")


Đối đầu với Quái vật Thông minh né tránh (Arena 30x30, 100 lượt chạy):
Thợ săn thất bại: 100/100 lượt (Tỷ lệ quái vật sống sót thành công: 100.0%)
